In [ ]:
from datetime import datetime
from itertools import chain
import pandas as pd
from bs4 import BeautifulSoup
from helpers import get_request

In [ ]:
def get_pages(num_pages: int):
    for i in range(num_pages):
        if i % 10 == 0:
            print("Getting page", i, "/", num_pages)

        try:
            response = get_request(f"https://www.ufc.com/trending/all?page={i}")
            yield response.text
        except Exception as e:
            print("Error getting page", i)
            print(e)

            yield None

In [3]:
# 200 pages is enough for about 8 months of articles
num_pages_to_get = 200
pages = [
    page for page in get_pages(num_pages_to_get)
    if page is not None
]
print("Got", len(pages), "pages")

Getting page 0 / 20
Getting page 10 / 20
Got 20 pages


In [4]:
def get_articles_data(page_content: str):
    soup = BeautifulSoup(page_content, "html.parser")
    articles = soup.find_all("a", "c-card--grid-card-trending")

    articles_data = [
        {
            "type": link.find_all("span", "c-card--grid-card-trending__info-prefix")[0].text,
            "time_ago": link.find_all("div", "field--name-datetime")[0].text,
            "headline": link.find_all("h3", "c-card--grid-card-trending__headline")[0].text.strip(),
            "link": link["href"]
        }
        for link in articles
    ]

    for data in articles_data:
        yield data

In [5]:
flattened_articles = list(chain.from_iterable(get_articles_data(page) for page in pages))
flattened_articles

[{'type': 'Interviews',
  'time_ago': '1 hour ago',
  'headline': 'Ciryl Gane Pre-Fight Interview | UFC 321',
  'link': '/video/150984'},
 {'type': 'Interviews',
  'time_ago': '1 hour ago',
  'headline': 'Tom Aspinall Pre-Fight Interview | UFC 321',
  'link': '/video/150983'},
 {'type': "Dana White's Contender Series",
  'time_ago': '3 hours ago',
  'headline': "Week 6 Preview | Dana White's Contender Series…",
  'link': '/news/week-6-preview-dana-whites-contender-series-season-9'},
 {'type': 'Announcements',
  'time_ago': '20 hours ago',
  'headline': 'Updates To UFC Vancouver',
  'link': '/news/updates-ufc-vancouver'},
 {'type': 'Highlights',
  'time_ago': '20 hours ago',
  'headline': 'Greatest Finishes | Noche UFC: Silva vs Lopes',
  'link': '/video/150976'},
 {'type': 'Fight Coverage',
  'time_ago': '20 hours ago',
  'headline': 'The Scorecard | Noche UFC',
  'link': '/news/scorecard-noche-ufc-san-antonio'},
 {'type': 'Athletes',
  'time_ago': '21 hours ago',
  'headline': 'KYLE D

In [6]:
articles_df = pd.DataFrame(data=flattened_articles).set_index("link")
articles_df

,type,time_ago,headline
link,,,
/video/150984,Interviews,1 hour ago,Ciryl Gane Pre-Fight Interview | UFC 321
/video/150983,Interviews,1 hour ago,Tom Aspinall Pre-Fight Interview | UFC 321
/news/week-6-preview-dana-whites-contender-series-season-9,Dana White's Contender Series,3 hours ago,Week 6 Preview | Dana White's Contender Series…
/news/updates-ufc-vancouver,Announcements,20 hours ago,Updates To UFC Vancouver
/video/150976,Highlights,20 hours ago,Greatest Finishes | Noche UFC: Silva vs Lopes
...,...,...,...
/news/full-undercard-confirmed-netflix-announces-broadcast-lineup-call-historic-canelo-vs-crawford,Announcements,3 weeks ago,Full Undercard Confirmed As Netflix Announces…
/news/aljamain-sterling-im-still-here-ufc-shanghai,Athletes,3 weeks ago,Aljamain Sterling: “I’m Still Here”
/video/150480,Interviews,3 weeks ago,Bruna Brasil Octagon Interview | Road To UFC…


In [7]:
filename = f"./article_links_{datetime.now().strftime('%d-%m-%y_%H-%M-%S')}.csv"
articles_df.to_csv(filename)